In [ ]:
import numpy as np
import scipy.io.wavfile as scipy_wav
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Audio
import pandas as pd
import math
import invrevsounds as IRS
sns.set_theme()

In [ ]:
def wide_plot(nrows=1, ncols=1):
    return plt.subplots(nrows, ncols, figsize=(12, 6))

In [ ]:
fig, axs = wide_plot(3, 1)
for idx in range(3):
    fsamps = IRS.float_data(f"samples/start-of-game-{idx}.wav")
    axs[idx].plot(fsamps)

In [ ]:
fig, axs = wide_plot(3, 1)
hpdata = []
for idx in range(3):
    fsamps = IRS.float_data(f"samples/start-of-game-{idx}.wav")
    xinfo = IRS.CrossingInfo.from_data(fsamps)
    axs[idx].plot(xinfo.half_periods)
    hpdata.append(xinfo.half_periods)

# Divide into chirps

Manually go through and look for breakpoints.  Some chirps look like they have a gap, some don't, so specify both ends of the range for all chirps.

In [ ]:
print([len(x) for x in hpdata])
hpdata[-1] = np.concat([hpdata[-1], [np.nan, np.nan]])

In [ ]:
fig, axs = wide_plot(3, 1)
for idx in range(3):
    slc = (214, 221)
    axs[idx].plot(np.arange(*slc, dtype=np.uint32), hpdata[idx][slc[0]:slc[1]], '.k')

In [ ]:
chirp_ranges = [
    [
        (0, 10), (11, 23), (24, 36), (37, 52), (53, 70), (71, 93), (94, 121), (122, 162), (164, 214),
        (214, 221), (222, 228), (229, 237), (238, 247), (247, 257), (258, 269), (270, 287), (288, 314)
    ],
    [
        (0, 10), (11, 23), (24, 36), (37, 52), (53, 70), (71, 93), (94, 121), (122, 162), (164, 214),
        (214, 221), (222, 228), (229, 237), (238, 247), (247, 257), (258, 269), (270, 287), (288, 314)
    ],
    [
        (0, 10), (11, 23), (24, 36), (37, 52), (53, 70), (71, 93), (94, 121), (122, 162), (164, 212),
        (212, 219), (220, 226), (227, 235), (236, 245), (245, 255), (256, 267), (268, 285), (286, 312)
    ]
]

In [ ]:
chirp_df = []
for si, (hps, rs) in enumerate(zip(hpdata, chirp_ranges)):
    for ci, r in enumerate(rs):
        chirp_hps = hps[r[0]:r[1]]
        chirp_xs = np.arange(len(chirp_hps), dtype=np.float64)
        lr = IRS.linregress(chirp_xs, chirp_hps)
        chirp_df.append((si, ci, r[0], r[1], np.sum(chirp_hps), lr.intercept, lr.slope, lr.rvalue))
chirp_df = pd.DataFrame.from_records(chirp_df, columns=["samp_idx", "chirp_idx", "hpi0", "hpi1", "dur", "hp0", "dhp", "rr"])
chirp_df

In [ ]:
avg_chirp = chirp_df[["chirp_idx", "dur", "hp0", "dhp"]].groupby(["chirp_idx"]).mean()
avg_chirp["dur"] *= (1e3 / IRS.FS_IN)
avg_chirp

In [ ]:
schp0 = IRS.linregress(np.arange(9), avg_chirp[:9]["hp0"])
schp1 = IRS.linregress(np.arange(8), avg_chirp[9:]["hp0"])

In [ ]:
all_dhps = avg_chirp[["dhp"]]
inlying_dhps = all_dhps[all_dhps > 0.6]
print("dhp in microseconds:", 1e6 * inlying_dhps.mean() / IRS.FS_IN)

In [ ]:
from matplotlib.lines import Line2D

fig, axs = wide_plot(3, 1)
axs[0].plot(avg_chirp["dur"], '.k')
axs[1].plot(avg_chirp["hp0"], '.k')
axs[1].add_line(Line2D([0, 8], [schp0.intercept, schp0.intercept + 8 * schp0.slope]))
axs[1].add_line(Line2D([9, 16], [schp1.intercept, schp1.intercept + 7 * schp0.slope]))
axs[2].plot(avg_chirp["dhp"], '.k')

mdur0 = avg_chirp["dur"][:9].mean()
print("mean duration first super-chirp in ms:", mdur0, "in stdchirps", 1e-3 * mdur0 / IRS.std_chirp_duration)
mdur1 = avg_chirp["dur"][9:].mean()
print("mean duration second super-chirp in ms:", mdur1, "in stdchirps", 1e-3 * mdur1 / IRS.std_chirp_duration)

In [ ]:
schp0_hp0_0 = schp0.intercept / IRS.FS_IN
schp0_d_hp0 = schp0.slope / IRS.FS_IN
schp1_hp0_0 = schp1.intercept / IRS.FS_IN
schp1_d_hp0 = schp1.slope / IRS.FS_IN

# Not sure about the apparent constant frequency first chirp in the second super-chirp.  Make it
# the same as the others.
#
# Did also experiment with putting the c.10ms gap in, but prefer without.
synth_schp0 = [IRS.Chirp(hp0, 14.72e-6, 1.68) for hp0 in np.linspace(schp0_hp0_0, schp0_hp0_0 + 8 * schp0_d_hp0, 9)]
synth_schp1 = [IRS.Chirp(hp0, 14.72e-6, 0.89) for hp0 in np.linspace(schp1_hp0_0, schp1_hp0_0 + 7 * schp1_d_hp0, 9)]

synth_chirps = (
    synth_schp0
    + synth_schp1
)

start_game = IRS.SoundEffect(synth_chirps)
start_game.write_wav(IRS.out_fname("start-of-game"), IRS.std_chirp_duration)
synth_samps = start_game.synthesised(IRS.std_chirp_duration)

fig, axs = wide_plot(2)
axs[0].plot(fsamps)
axs[1].plot(synth_samps);

In [ ]:
Audio(synth_samps, rate=IRS.FS_OUT)

In [ ]:
Audio(fsamps, rate=IRS.FS_IN)